Cell 1 — setup

In [5]:
import os, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid", palette="colorblind")

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
processed = PROJECT_ROOT / "data" / "processed"
reports   = PROJECT_ROOT / "reports"; reports.mkdir(exist_ok=True)
models    = PROJECT_ROOT / "models";  models.mkdir(exist_ok=True)
figdir    = PROJECT_ROOT / "figures"

RANDOM_STATE = 42
train = pd.read_csv(processed / "master_train.csv")
test  = pd.read_csv(processed  / "master_test.csv")

Cell 2 — feature lists

In [6]:
num_cols = ["altitude_m","rainfall_mm_season","farm_size_ha","fertilizer_kg_per_ha",
            "soil_quality_index","labor_days_per_ha","distance_to_market_km",
            "season_mean_temp","season_total_rain","season_extreme_heat",
            "season_months_count","temp_dev","rain_dev","fert_x_seed","planting_month_num"]
weather_cols = ["season_mean_temp","season_total_rain","season_extreme_heat",
                "season_months_count","temp_dev","rain_dev","region_year_temp","region_year_rain"]
cat_cols = ["region","crop_type"]
features = num_cols + cat_cols
X = train[features]; y = train["yield_tons_per_ha"]

Cell 3 — D1 baselines

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def make_pre(cols_num, cols_cat):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc",  StandardScaler())]), cols_num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cols_cat),
    ])

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

def evaluate(model, X_tr, y_tr, X_val, y_val):
    t0=time.time(); model.fit(X_tr, y_tr); t=time.time()-t0
    p = model.predict(X_val)
    return dict(rmse=float(np.sqrt(mean_squared_error(y_val,p))),
                mae=float(mean_absolute_error(y_val,p)),
                r2=float(r2_score(y_val,p)), seconds=round(t,2))

results = {}
results["Mean baseline"] = evaluate(Pipeline([("d", DummyRegressor(strategy="mean"))]), X_tr,y_tr,X_val,y_val)
results["Ridge (linear)"] = evaluate(Pipeline([("pre", make_pre(num_cols, cat_cols)),
                                               ("reg", Ridge(alpha=1.0, random_state=RANDOM_STATE))]),
                                     X_tr,y_tr,X_val,y_val)
print(pd.DataFrame(results).T)

                    rmse       mae        r2  seconds
Mean baseline   1.413051  1.104234 -0.000113     0.02
Ridge (linear)  0.952845  0.722412  0.545245     0.03


Cell 4 — D2 model comparison

In [8]:
models_dict = {
    "RandomForest": Pipeline([("pre", make_pre(num_cols, cat_cols)),
                              ("reg", RandomForestRegressor(n_estimators=300,
                                                            random_state=RANDOM_STATE, n_jobs=-1))]),
    "HistGradientBoosting": Pipeline([("pre", make_pre(num_cols, cat_cols)),
                                      ("reg", HistGradientBoostingRegressor(random_state=RANDOM_STATE))]),
    "Ridge": Pipeline([("pre", make_pre(num_cols, cat_cols)),
                       ("reg", Ridge(alpha=1.0, random_state=RANDOM_STATE))]),
}
for name, m in models_dict.items():
    results[name] = evaluate(m, X_tr, y_tr, X_val, y_val)
print(pd.DataFrame(results).T)

# Save comparison for fig10
comp_path = reports / "model_comparison.json"
prev = json.loads(comp_path.read_text()) if comp_path.exists() else {}
prev.update({k: v for k, v in results.items()})
prev["__baseline__"] = results["Mean baseline"]
comp_path.write_text(json.dumps(prev, indent=2))

                          rmse       mae        r2  seconds
Mean baseline         1.413051  1.104234 -0.000113     0.02
Ridge (linear)        0.952845  0.722412  0.545245     0.03
RandomForest          0.634654  0.470067  0.798253     2.44
HistGradientBoosting  0.613830  0.457823  0.811275     0.36
Ridge                 0.952845  0.722412  0.545245     0.04


842

Cell 5 — D3 cross-validation (final + runner-up)

In [9]:
from sklearn.model_selection import KFold, cross_val_score

final = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
runner = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)

pipe_final = Pipeline([("pre", make_pre(num_cols, cat_cols)), ("reg", final)])
pipe_runner = Pipeline([("pre", make_pre(num_cols, cat_cols)), ("reg", runner)])

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rmse_final = -cross_val_score(pipe_final, X, y, cv=kf, scoring="neg_root_mean_squared_error")
rmse_runner = -cross_val_score(pipe_runner, X, y, cv=kf, scoring="neg_root_mean_squared_error")
print(f"Final  HGB  CV RMSE: {rmse_final.mean():.4f} ± {rmse_final.std():.4f}")
print(f"Runner RF   CV RMSE: {rmse_runner.mean():.4f} ± {rmse_runner.std():.4f}")

comp = json.loads(comp_path.read_text())
comp["HistGradientBoosting"]["rmse_std"] = float(rmse_final.std())
comp["RandomForest"]["rmse_std"] = float(rmse_runner.std())
comp_path.write_text(json.dumps(comp, indent=2))

Final  HGB  CV RMSE: 0.6214 ± 0.0141
Runner RF   CV RMSE: 0.6423 ± 0.0094


918

Cell 6 — D4 out-of-time check

In [10]:
tr = train[train["survey_year"] <= 2023]
va = train[train["survey_year"] == 2024]
ot = evaluate(Pipeline([("pre", make_pre(num_cols, cat_cols)),
                        ("reg", HistGradientBoostingRegressor(random_state=RANDOM_STATE))]),
               tr[features], tr["yield_tons_per_ha"],
               va[features], va["yield_tons_per_ha"])
print("Out-of-time (train 2021-23, val 2024):", ot)

Out-of-time (train 2021-23, val 2024): {'rmse': 0.6501945840668691, 'mae': 0.48697204897804275, 'r2': 0.7944804293635741, 'seconds': 0.25}


Cell 7 — D5 weather ablation

In [11]:
num_no_weather = [c for c in num_cols if c not in weather_cols]
features_no_weather = num_no_weather + cat_cols

def build(cols_num, cols_cat):
    return Pipeline([("pre", make_pre(cols_num, cols_cat)),
                     ("reg", HistGradientBoostingRegressor(random_state=RANDOM_STATE))])

no_w  = evaluate(build(num_no_weather, cat_cols),
                 train[features_no_weather], y,
                 train[features_no_weather], y)
# Use same split as earlier:
X_tr_nw = X_tr[num_no_weather + cat_cols]; X_val_nw = X_val[num_no_weather + cat_cols]
with_w = evaluate(build(num_cols, cat_cols), X_tr, y_tr, X_val, y_val)
no_w   = evaluate(build(num_no_weather, cat_cols), X_tr_nw, y_tr, X_val_nw, y_val)
print("Without weather:", no_w)
print("With    weather:", with_w)
print("Delta RMSE:", round(no_w["rmse"] - with_w["rmse"], 4))

Without weather: {'rmse': 0.678726974412663, 'mae': 0.4987149995286521, 'r2': 0.76925955360148, 'seconds': 0.26}
With    weather: {'rmse': 0.6138299873657616, 'mae': 0.4578234467163912, 'r2': 0.8112748793429428, 'seconds': 0.3}
Delta RMSE: 0.0649


Cell 8 — D6 tuning (small grid)

In [12]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

search = RandomizedSearchCV(
    Pipeline([("pre", make_pre(num_cols, cat_cols)),
              ("reg", HistGradientBoostingRegressor(random_state=RANDOM_STATE))]),
    param_distributions={
        "reg__learning_rate": uniform(0.03, 0.15),
        "reg__max_iter": randint(150, 500),
        "reg__max_leaf_nodes": randint(15, 60),
        "reg__l2_regularization": uniform(0.0, 1.0),
    },
    n_iter=20, cv=3, scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1,
)
search.fit(X_tr, y_tr)
print("Best params:", search.best_params_)
print("Best CV RMSE:", -search.best_score_)
best_model = search.best_estimator_
p = best_model.predict(X_val)
print("Tuned val RMSE:", float(np.sqrt(mean_squared_error(y_val, p))))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best params: {'reg__l2_regularization': np.float64(0.8021969807540397), 'reg__learning_rate': np.float64(0.04118259655196562), 'reg__max_iter': 476, 'reg__max_leaf_nodes': 23}
Best CV RMSE: 0.6218638493997948
Tuned val RMSE: 0.6106610185876278


Cell 9 — Save validation preds for fig11

In [13]:
val_preds = pd.DataFrame({"y_true": y_val.values, "y_pred": p})
val_preds.to_csv(reports / "validation_preds.csv", index=False)
print(val_preds.head())

     y_true    y_pred
0  5.029958  5.432403
1  3.901528  3.678193
2  1.544510  1.133080
3  5.179462  3.718058
4  3.852299  3.624170



Cell 10 — D7 error analysis


In [14]:
val_tbl = X_val.copy().reset_index(drop=True)
val_tbl["y_true"] = y_val.values; val_tbl["y_pred"] = p
val_tbl["err"] = val_tbl["y_true"] - val_tbl["y_pred"]
val_tbl["abs_err"] = val_tbl["err"].abs()

print("By crop:\n", val_tbl.groupby("crop_type")["abs_err"].agg(["mean","count"]).round(3))
print("\nBy region:\n", val_tbl.groupby("region")["abs_err"].agg(["mean","count"]).round(3))
print("\nWorst 10:\n", val_tbl.sort_values("abs_err", ascending=False).head(10)[
    ["plot_id" if "plot_id" in val_tbl.columns else "y_true","y_true","y_pred","err","crop_type","region"]])

By crop:
             mean  count
crop_type              
barley     0.387    620
maize      0.628    593
sorghum    0.429    590
teff       0.345    604
wheat      0.483    611

By region:
          mean  count
region              
Amhara  0.520    594
Oromia  0.498    589
SNNPR   0.478    569
Somali  0.259    632
Tigray  0.521    634

Worst 10:
         y_true    y_true    y_pred       err crop_type  region
2141  2.852801  2.852801  6.325666 -3.472865     maize   SNNPR
335   8.924457  8.924457  6.139017  2.785440     maize  Tigray
440   7.413014  7.413014  4.685277  2.727737     maize  Tigray
43    6.970028  6.970028  4.322335  2.647693     wheat  Tigray
1965  7.035693  7.035693  4.475641  2.560052     wheat  Amhara
1783  0.428278  0.428278  2.988247 -2.559969   sorghum  Tigray
2447  8.283826  8.283826  5.752086  2.531740     maize  Tigray
1045  8.211242  8.211242  5.746650  2.464592     wheat  Amhara
1944  0.851374  0.851374  3.230966 -2.379592     maize  Amhara
1155  5.198916  5.19

Cell 11 — D8 save importance

In [15]:
from sklearn.inspection import permutation_importance
sample = X_val.sample(min(1500, len(X_val)), random_state=RANDOM_STATE)
r = permutation_importance(best_model, sample, y_val.loc[sample.index],
                           n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.DataFrame({"feature": X_val.columns, "importance": r.importances_mean})
imp.sort_values("importance", ascending=False).to_csv(reports / "feature_importance.csv", index=False)
print(imp.sort_values("importance", ascending=False).head(10))

                 feature  importance
16             crop_type    0.808395
7       season_mean_temp    0.635228
0             altitude_m    0.587180
4     soil_quality_index    0.092134
13           fert_x_seed    0.078443
3   fertilizer_kg_per_ha    0.043336
1     rainfall_mm_season    0.016308
5      labor_days_per_ha    0.004832
9    season_extreme_heat    0.001265
14    planting_month_num    0.000071


Cell 12 — Save final model + submission

In [16]:
from pathlib import Path

print("BEFORE fix, PROJECT_ROOT =", globals().get("PROJECT_ROOT"))

PROJECT_ROOT = Path(r"C:\Users\sclab\Desktop\A hackaton")

print("AFTER  fix, PROJECT_ROOT =", PROJECT_ROOT)
print("exists:", PROJECT_ROOT.exists())
print("template exists:", (PROJECT_ROOT / "data" / "raw" / "submission_template.csv").exists())

BEFORE fix, PROJECT_ROOT = c:\Users\sclab\Desktop\A hackaton\notebooks
AFTER  fix, PROJECT_ROOT = C:\Users\sclab\Desktop\A hackaton
exists: True
template exists: True


In [17]:
import joblib
import pandas as pd

# (assumes best_model, features, X, y, test, PROJECT_ROOT, models already defined)
best_model.fit(X, y)                                     # refit on full data
models.mkdir(exist_ok=True)
joblib.dump(best_model, models / "final_model.joblib")

template = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "submission_template.csv")
test_pred = best_model.predict(test[features])

sub = template[["plot_id"]].merge(
    pd.DataFrame({"plot_id": test["plot_id"].values, "yield_tons_per_ha": test_pred}),
    on="plot_id", how="left",
)
sub.to_csv(PROJECT_ROOT / "submission" / "team_dawit_submission.csv", index=False)
print(sub.head(), sub.shape)

       plot_id  yield_tons_per_ha
0  PLOT-017296           3.357822
1  PLOT-017259           2.211954
2  PLOT-016178           2.906590
3  PLOT-018610           2.856376
4  PLOT-015109           2.304373 (3750, 2)


Cell 13 — D9 plain language

In [18]:
rmse = float(np.sqrt(mean_squared_error(y_val, p)))
mae  = float(mean_absolute_error(y_val, p))
mean_y = y.mean()
prices_avg = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "market_prices.csv")["price_birr_per_quintal"].mean()
print(f"RMSE = {rmse:.2f} t/ha ({rmse/mean_y*100:.1f}% of mean yield)")
print(f"MAE  = {mae:.2f} t/ha ({mae/mean_y*100:.1f}% of mean yield)")
print(f"Approx MAE per hectare in birr: {mae*10*prices_avg:,.0f}")

RMSE = 0.61 t/ha (22.1% of mean yield)
MAE  = 0.45 t/ha (16.4% of mean yield)
Approx MAE per hectare in birr: 20,161


In [19]:
# --- Export app assets (app reads these at runtime) ---
from pathlib import Path
import pandas as pd

# Paths (PROJECT_ROOT / processed should already exist from earlier cells)
assets = PROJECT_ROOT / "app" / "assets"
assets.mkdir(parents=True, exist_ok=True)

# --- weather ---
weather_raw = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "regional_weather.csv")

def clean_region(x):
    x = str(x).strip().lower()
    if x.startswith("orom") or x == "oro":  return "Oromia"
    if x.startswith("amha") or x == "amh":  return "Amhara"
    if x.startswith("snn"):                 return "SNNPR"
    if x.startswith("tigr") or x == "tig":  return "Tigray"
    if x.startswith("soma") or x == "som":  return "Somali"
    return x.title()

weather_raw["region"] = weather_raw["region"].apply(clean_region)
mn = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
      "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}
weather_raw["month"] = (weather_raw["month"].astype(str)
                        .str.strip().str[:3].str.lower().map(mn))
for c in ["avg_temp_c", "monthly_rainfall_mm", "extreme_heat_days"]:
    weather_raw[c] = pd.to_numeric(weather_raw[c], errors="coerce")
weather_raw = weather_raw.dropna(subset=["month"]).drop_duplicates(
    subset=["region","year","month"], keep="first")
weather_raw[["region","year","month","avg_temp_c",
             "monthly_rainfall_mm","extreme_heat_days"]] \
    .to_csv(assets / "season_weather.csv", index=False)

# --- prices ---
prices_raw = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "market_prices.csv")
prices_raw["region"]    = prices_raw["region"].apply(clean_region)
prices_raw["crop_type"] = (prices_raw["crop_type"].astype(str)
                           .str.strip().str.lower()
                           .replace({"tef": "teff"}))
prices_raw.to_csv(assets / "price_table.csv", index=False)

# --- regional averages from master_train ---
train_tmp = pd.read_csv(processed / "master_train.csv")
avg = (train_tmp.groupby(["region","crop_type"])["yield_tons_per_ha"]
       .mean().reset_index()
       .rename(columns={"yield_tons_per_ha": "mean_yield"}))
avg.to_csv(assets / "region_crop_avg.csv", index=False)

print("Assets written:", [p.name for p in assets.glob("*.csv")])

Assets written: ['price_table.csv', 'region_crop_avg.csv', 'season_weather.csv']


In [20]:
# ---- Export everything the Gradio app needs into app/assets/ ----
import shutil, joblib
from pathlib import Path
import pandas as pd

assets = PROJECT_ROOT / "app" / "assets"
assets.mkdir(parents=True, exist_ok=True)

# 1. Save the model directly into app/assets/
joblib.dump(best_model, assets / "final_model.joblib")

# 2. (also keep a copy in models/ for reference)
(PROJECT_ROOT / "models").mkdir(exist_ok=True)
joblib.dump(best_model, PROJECT_ROOT / "models" / "final_model.joblib")

# 3. Region cleaning helper
def clean_region(x):
    x = str(x).strip().lower()
    if x.startswith("orom") or x == "oro":  return "Oromia"
    if x.startswith("amha") or x == "amh":  return "Amhara"
    if x.startswith("snn"):                 return "SNNPR"
    if x.startswith("tigr") or x == "tig":  return "Tigray"
    if x.startswith("soma") or x == "som":  return "Somali"
    return x.title()

# 4. Weather asset
weather_raw = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "regional_weather.csv")
weather_raw["region"] = weather_raw["region"].apply(clean_region)
mn = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
      "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}
weather_raw["month"] = (weather_raw["month"].astype(str)
                        .str.strip().str[:3].str.lower().map(mn))
for c in ["avg_temp_c","monthly_rainfall_mm","extreme_heat_days"]:
    weather_raw[c] = pd.to_numeric(weather_raw[c], errors="coerce")
weather_raw = weather_raw.dropna(subset=["month"]).drop_duplicates(
    subset=["region","year","month"], keep="first")
weather_raw[["region","year","month","avg_temp_c",
             "monthly_rainfall_mm","extreme_heat_days"]] \
    .to_csv(assets / "season_weather.csv", index=False)

# 5. Price asset
prices_raw = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "market_prices.csv")
prices_raw["region"]    = prices_raw["region"].apply(clean_region)
prices_raw["crop_type"] = (prices_raw["crop_type"].astype(str)
                           .str.strip().str.lower().replace({"tef":"teff"}))
prices_raw.to_csv(assets / "price_table.csv", index=False)

# 6. Region-crop averages
train_tmp = pd.read_csv(processed / "master_train.csv")
avg = (train_tmp.groupby(["region","crop_type"])["yield_tons_per_ha"]
       .mean().reset_index().rename(columns={"yield_tons_per_ha":"mean_yield"}))
avg.to_csv(assets / "region_crop_avg.csv", index=False)

# 7. Verify
print("Assets in app/assets/:")
for f in sorted(assets.glob("*")):
    print("  ", f.name, f.stat().st_size, "bytes")

Assets in app/assets/:
   final_model.joblib 828258 bytes
   price_table.csv 2634 bytes
   region_crop_avg.csv 846 bytes
   season_weather.csv 6260 bytes


In [21]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

num_cols = [c for c in X.columns if c not in ("region","crop_type")]
cat_cols = ["region","crop_type"]

pre = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
best_model = Pipeline([("pre", pre), ("model", HistGradientBoostingRegressor())])
best_model.fit(X, y)     # X includes region/crop_type as strings

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('pre', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['altitude_m','rainfall_mm_season','farm_size_ha',...,'planting_month_num', 'region','crop_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This 

In [22]:
from pathlib import Path

ROOT = Path(r"C:\Users\sclab\Desktop\A hackaton")

for p in sorted(ROOT.rglob("*")):
    # skip the noise
    if any(x in p.parts for x in [".venv", "venv", "__pycache__", ".ipynb_checkpoints", ".git"]):
        continue
    rel = p.relative_to(ROOT)
    depth = len(rel.parts) - 1
    mark = "DIR " if p.is_dir() else "file"
    print(f"{'  ' * depth}{mark} {p.name}")

file .gitignore
DIR  .gradio
  DIR  flagged
    file dataset1.csv
DIR  app
  file app.py
  DIR  assets
    file final_model.joblib
    file price_table.csv
    file region_crop_avg.csv
    file season_weather.csv
  file requirements.txt
DIR  data
  DIR  processed
    file data_dictionary_master.csv
    file master_test.csv
    file master_train.csv
  DIR  raw
    file crop_yield_leaderboard_test.csv
    file crop_yield_train.csv
    file market_prices.csv
    file regional_weather.csv
    file submission_template.csv
DIR  figures
  file fig01_missingness.png
  file fig02_before_after_cleaning.png
  file fig03_yield_distribution.png
  file fig04_region_crop_heatmap.png
  file fig05_correlation_heatmap.png
  file fig06_climate_by_region.png
  file fig07_yield_vs_season_temp.png
  file fig08_price_trends.png
  file fig09_revenue_by_crop_region.png
  file figure_captions.md
DIR  models
  file final_model.joblib
DIR  notebooks
  DIR  data
    DIR  processed
      file data_dictionary_master

In [ ]:
from pathlib import Path
import shutil

ROOT = Path(r"C:\Users\sclab\Desktop\A hackaton")
NB = ROOT / "notebooks"

# 1. Move the four notebooks up from notebooks\notebooks\ to notebooks\
inner = NB / "notebooks"
if inner.exists():
    for ipynb in inner.glob("*.ipynb"):
        target = NB / ipynb.name
        if target.exists():
            print(f"SKIP (already exists): {ipynb.name}")
        else:
            shutil.move(str(ipynb), str(target))
            print(f"moved: {ipynb.name}")
    # remove the now-empty inner folder
    try:
        inner.rmdir()
        print("removed empty:", inner)
    except OSError as e:
        print("could not rmdir (not empty?):", inner, "->", e)

# 2. Save the 3 extra useful files from notebooks\reports\ before deleting
src_reports = NB / "reports"
dst_reports = ROOT / "reports"
dst_reports.mkdir(exist_ok=True)
for name in ["feature_importance.csv", "model_comparison.json", "validation_preds.csv"]:
    src = src_reports / name
    dst = dst_reports / name
    if src.exists() and not dst.exists():
        shutil.copy(str(src), str(dst))
        print(f"copied: {name} -> reports\\")

# 3. Delete the duplicate tree inside notebooks\
for sub in ["data", "figures", "models", "reports", "submission"]:
    p = NB / sub
    if p.exists():
        shutil.rmtree(p)
        print(f"deleted duplicate: {p.relative_to(ROOT)}")

# 4. Delete the .gradio junk
g = ROOT / ".gradio"
if g.exists():
    shutil.rmtree(g)
    print("deleted:", g.relative_to(ROOT))

# 5. Delete the truncated presentation file
p = ROOT / "presentation" / "p"
if p.exists():
    p.unlink()
    print("deleted:", p.relative_to(ROOT))

# 6. Show the final tree (excluding noise)
print("\n--- final tree ---")
for p in sorted(ROOT.rglob("*")):
    if any(x in p.parts for x in [".venv", "venv", "__pycache__",
                                   ".ipynb_checkpoints", ".git", ".gradio"]):
        continue
    rel = p.relative_to(ROOT)
    depth = len(rel.parts) - 1
    kind = "DIR " if p.is_dir() else "file"
    print(f"{'  ' * depth}{kind} {p.name}")

In [1]:
from pathlib import Path
ROOT = Path(r"C:\Users\sclab\Desktop\A hackaton")

# Top-level files
(ROOT / "README.md").touch()
(ROOT / "requirements.txt").touch()

# Report markdown files
for f in ["A_cleaning_and_integration.md", "B_analysis_report.md", "D_model_evaluation.md"]:
    (ROOT / "reports" / f).touch()

# Slides file
(ROOT / "presentation" / "team_dawit_slides.md").touch()

# Optional src files
for f in ["cleaning.py", "features.py", "train.py", "predict.py"]:
    (ROOT / "src" / f).touch()

# .gitignore
(ROOT / ".gitignore").write_text(
    ".venv/\nvenv/\n__pycache__/\n.ipynb_checkpoints/\n"
    ".gradio/\n*.pyc\n.DS_Store\n"
)

print("placeholder files created")

placeholder files created


In [2]:
from pathlib import Path

ROOT = Path(r"C:\Users\sclab\Desktop\A hackaton")

for p in sorted(ROOT.rglob("*")):
    # skip the noise
    if any(x in p.parts for x in [".venv", "venv", "__pycache__", ".ipynb_checkpoints", ".git"]):
        continue
    rel = p.relative_to(ROOT)
    depth = len(rel.parts) - 1
    mark = "DIR " if p.is_dir() else "file"
    print(f"{'  ' * depth}{mark} {p.name}")

file .gitignore
DIR  app
  file app.py
  DIR  assets
    file final_model.joblib
    file price_table.csv
    file region_crop_avg.csv
    file season_weather.csv
  file requirements.txt
DIR  data
  DIR  processed
    file data_dictionary_master.csv
    file master_test.csv
    file master_train.csv
  DIR  raw
    file crop_yield_leaderboard_test.csv
    file crop_yield_train.csv
    file market_prices.csv
    file regional_weather.csv
    file submission_template.csv
DIR  figures
  file fig01_missingness.png
  file fig02_before_after_cleaning.png
  file fig03_yield_distribution.png
  file fig04_region_crop_heatmap.png
  file fig05_correlation_heatmap.png
  file fig06_climate_by_region.png
  file fig07_yield_vs_season_temp.png
  file fig08_price_trends.png
  file fig09_revenue_by_crop_region.png
  file figure_captions.md
DIR  models
  file final_model.joblib
DIR  notebooks
  file 01_cleaning_and_integration.ipynb
  file 02_analysis_report.ipynb
  file 03_visualizations.ipynb
  file 04_

In [1]:
from pathlib import Path
import shutil, os, stat

ROOT = Path(r"C:\Users\sclab\Desktop\A hackaton")
inner = ROOT / "notebooks" / "notebooks"

def on_rm_error(func, path, exc_info):
    os.chmod(path, stat.S_IWRITE)
    func(path)

if inner.exists():
    shutil.rmtree(inner, onerror=on_rm_error)
    print("deleted:", inner)
else:
    print("already gone")

deleted: C:\Users\sclab\Desktop\A hackaton\notebooks\notebooks
